# Huấn luyện mô hình dự đoán hiệu năng Gaming

Notebook huấn luyện mô hình hồi quy log–log để dự đoán **3DMark Time Spy Overall** từ hai điểm tham chiếu:

- **GpuScore:** PassMark G3D Mark của GPU, đúng phiên bản dung lượng VRAM.
- **CpuMultiScore:** PassMark CPU Mark của CPU.

Mô hình có dạng:

`Score = A × GpuScore^b × CpuMultiScore^c`

Các hệ số `b` và `c` được ràng buộc không âm để điểm dự đoán không giảm khi tăng từng điểm đầu vào, giữ điểm còn lại cố định.

Phương án này được chọn sau vòng so sánh với hồi quy có tương tác CPU–GPU. Notebook này huấn luyện lại trên toàn bộ dữ liệu hợp lệ và xuất hệ số để tích hợp backend. Kết quả đánh giá CV/test được lưu ở notebook so sánh riêng; sai số trên dữ liệu dùng để huấn luyện không được coi là đánh giá độc lập.

Điểm dự đoán là điểm benchmark Time Spy, không phải FPS của từng trò chơi. Hệ thống tư vấn cần kết hợp điểm này với ngân sách, tương thích linh kiện và yêu cầu sử dụng.

## 1. Đọc dữ liệu từ Excel

Tải lên file Excel đã chỉnh sửa và đọc hai sheet:

- `Benchmark_Gaming`: các lượt chạy benchmark, cấu hình máy và điểm Time Spy.
- `ReferenceBenchmark`: điểm PassMark dùng làm đầu vào cho mô hình.

Bước này chỉ đọc dữ liệu, kiểm tra tên sheet và thống kê trạng thái duyệt. Chưa loại dòng, ghép điểm hoặc huấn luyện mô hình.

Mã SHA256 được lưu để nhận diện chính xác phiên bản file Excel đã sử dụng.

In [ ]:
import io
import hashlib
import pandas as pd

from google.colab import files
from IPython.display import display

# Tải lên đúng một file Excel.
uploaded = files.upload()

excel_names = [
    name for name in uploaded
    if name.lower().endswith(".xlsx")
]

if len(excel_names) != 1:
    raise ValueError("Hãy tải lên đúng một file Excel .xlsx.")

excel_name = excel_names[0]
excel_bytes = uploaded[excel_name]
excel_sha256 = hashlib.sha256(excel_bytes).hexdigest()

# Đọc và kiểm tra hai sheet cần thiết.
sheets = pd.read_excel(io.BytesIO(excel_bytes), sheet_name=None)

required_sheets = ["Benchmark_Gaming", "ReferenceBenchmark"]
missing_sheets = [
    name for name in required_sheets if name not in sheets
]

if missing_sheets:
    raise ValueError(f"File thiếu các sheet: {missing_sheets}")

gaming_raw = sheets["Benchmark_Gaming"].copy()
reference_raw = sheets["ReferenceBenchmark"].copy()

for frame in (gaming_raw, reference_raw):
    frame.columns = frame.columns.str.strip()

if "ReviewStatus" not in gaming_raw.columns:
    raise ValueError("Sheet Benchmark_Gaming thiếu cột ReviewStatus.")

# Chỉ thống kê, chưa lọc dữ liệu.
print("File:", excel_name)
print("SHA256:", excel_sha256)
print("Số dòng Gaming:", len(gaming_raw))
print("Số dòng Reference:", len(reference_raw))

print("\nTrạng thái duyệt của dữ liệu Gaming:")
display(
    gaming_raw["ReviewStatus"]
    .value_counts(dropna=False)
    .rename_axis("ReviewStatus")
    .to_frame("Số dòng")
)

Saving Benchmark (6) (1).xlsx to Benchmark (6) (1).xlsx
File: Benchmark (6) (1).xlsx
SHA256: 2d3e796ca57f027c86cdfb5598d61390d4e07c1eb0acca86ee225e2948666895
Số dòng Gaming: 146
Số dòng Reference: 41

Trạng thái duyệt của dữ liệu Gaming:


,Số dòng
ReviewStatus,
ACCEPT,144
REVIEW,2


## 2. Chuẩn bị dữ liệu đầu vào

Chỉ sử dụng các lượt benchmark có trạng thái `ACCEPT`, bài kiểm tra `Time Spy` và chỉ số `Overall`. Các dòng còn lại được lưu riêng để theo dõi.

Ghép hai điểm tham chiếu từ sheet `ReferenceBenchmark`:

- `CpuMultiScore`: CPU Mark của CPU.
- `GpuScore`: G3D Mark của GPU.

Tên linh kiện được chuẩn hóa để xử lý khác biệt cách viết như khoảng trắng, dấu gạch nối, từ “Processor” và tiền tố “NVIDIA”. Mã linh kiện, hậu tố và dung lượng VRAM được giữ nguyên; không sử dụng ghép gần đúng.

Dừng xử lý nếu mã lượt chạy hoặc nguồn benchmark bị thiếu/trùng, điểm tham chiếu không xác định duy nhất, hoặc điểm đầu vào/điểm đích không phải số hữu hạn lớn hơn 0. Không thay dữ liệu thiếu bằng 0.

RAM không nằm trong công thức của mô hình này. Bước chuẩn bị vẫn giữ các cột gốc để truy vết dữ liệu.

In [ ]:
import re
import numpy as np

# Làm việc trên bản sao, giữ nguyên dữ liệu đã đọc từ Excel.
gaming = gaming_raw.copy()
reference = reference_raw.copy()

required_gaming = [
    "RunId", "CpuModel", "GpuModel", "TestName",
    "Metric", "Score", "SourceUrl", "ReviewStatus"
]
required_reference = [
    "ReferenceId", "ComponentType", "ModelName",
    "TestName", "Metric", "RawScore", "ReviewStatus"
]

for label, frame, columns in [
    ("Gaming", gaming, required_gaming),
    ("Reference", reference, required_reference)
]:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} thiếu cột: {missing}")

    # Bỏ khoảng trắng thừa, giữ ô thiếu ở dạng thiếu.
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        frame[column] = frame[column].astype("string").str.strip()


# 1. Lọc đúng loại benchmark và trạng thái duyệt.
eligible = (
    gaming["ReviewStatus"].str.upper().eq("ACCEPT")
    & gaming["TestName"].eq("Time Spy")
    & gaming["Metric"].eq("Overall")
).fillna(False)

excluded_rows = gaming.loc[~eligible].copy()
data = gaming.loc[eligible].copy()

if data.empty:
    raise ValueError("Không có dòng Gaming hợp lệ để huấn luyện.")

for column in ["RunId", "SourceUrl"]:
    invalid = (
        data[column].isna()
        | data[column].eq("")
        | data[column].duplicated(keep=False)
    )
    if invalid.any():
        display(data.loc[invalid, ["RunId", "SourceUrl"]])
        raise ValueError(f"{column} bị thiếu hoặc trùng; cần kiểm tra.")


# 2. Chuẩn hóa tên nhưng giữ mã CPU/GPU và dung lượng VRAM.
def component_key(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().split("@")[0]
    text = re.sub(r"\b(processor|nvidia)\b", "", text)
    return re.sub(r"[^a-z0-9]", "", text)


data["CpuKey"] = data["CpuModel"].map(component_key)
data["GpuKey"] = data["GpuModel"].map(component_key)
reference["ComponentKey"] = reference["ModelName"].map(component_key)


# 3. Ghép điểm theo đúng linh kiện, bài kiểm tra và loại điểm.
mapping_specs = [
    ("CPU", "CPU Mark", "CpuKey", "CpuMultiScore", "CpuMultiRefId"),
    ("GPU", "G3D Mark", "GpuKey", "GpuScore", "GpuRefId")
]

for component, metric, key_column, score_column, id_column in mapping_specs:
    accepted_reference = (
        reference["ComponentType"].eq(component)
        & reference["TestName"].eq(f"PassMark {component}")
        & reference["Metric"].eq(metric)
        & reference["ReviewStatus"].str.upper().eq("ACCEPT")
    ).fillna(False)

    selected = reference.loc[accepted_reference].copy()

    if selected.empty:
        raise ValueError(f"Không tìm thấy điểm tham chiếu: {metric}.")

    bad_reference = (
        selected["ComponentKey"].eq("")
        | selected["ComponentKey"].duplicated(keep=False)
        | selected["ReferenceId"].isna()
        | selected["ReferenceId"].eq("")
        | selected["ReferenceId"].duplicated(keep=False)
    )
    if bad_reference.any():
        display(selected.loc[bad_reference])
        raise ValueError(f"Tham chiếu {metric} bị thiếu hoặc không duy nhất.")

    lookup = selected.set_index("ComponentKey")
    matched_ids = data[key_column].map(lookup["ReferenceId"])
    matched_scores = data[key_column].map(lookup["RawScore"])

    if matched_ids.isna().any():
        display(data.loc[
            matched_ids.isna(), ["RunId", "CpuModel", "GpuModel"]
        ])
        raise ValueError(f"Có linh kiện chưa ghép được điểm {metric}.")

    # Nếu Excel đã ghi mã tham chiếu, kiểm tra mã đó có khớp tên không.
    if id_column in data.columns:
        existing_ids = data[id_column].astype("string").str.strip()
        supplied = existing_ids.notna() & existing_ids.ne("")
        mismatch = (
            supplied & existing_ids.ne(matched_ids.astype("string"))
        ).fillna(False)

        if mismatch.any():
            display(data.loc[
                mismatch, ["RunId", "CpuModel", "GpuModel", id_column]
            ])
            raise ValueError(f"{id_column} mâu thuẫn với tên linh kiện.")

    data[id_column] = matched_ids
    data[score_column] = matched_scores


# 4. Log chỉ nhận điểm hữu hạn và lớn hơn 0.
numeric_columns = ["GpuScore", "CpuMultiScore", "Score"]

for column in numeric_columns:
    data[column] = pd.to_numeric(
        data[column], errors="coerce"
    ).astype(float)

invalid_numeric = (
    ~np.isfinite(data[numeric_columns])
    | data[numeric_columns].le(0)
).any(axis=1)

if invalid_numeric.any():
    display(data.loc[
        invalid_numeric, ["RunId"] + numeric_columns
    ])
    raise ValueError("Có điểm bị thiếu, không hữu hạn hoặc không lớn hơn 0.")

data = data.sort_values("RunId").reset_index(drop=True)


# 5. Hiển thị kết quả chuẩn bị dữ liệu.
print("Số dòng được dùng:", len(data))
print("Số dòng không dùng:", len(excluded_rows))
print("Số CPU:", data["CpuKey"].nunique())
print("Số GPU:", data["GpuKey"].nunique())

print("\nCác dòng không dùng:")
display(excluded_rows[["RunId", "ReviewStatus", "TestName", "Metric"]])

print("\nVí dụ dữ liệu đã ghép điểm:")
display(data[
    ["RunId", "CpuModel", "GpuModel",
     "CpuMultiScore", "GpuScore", "Score"]
].head())

print("\nKhoảng điểm trong dữ liệu:")
display(data[numeric_columns].agg(["min", "max"]))

Số dòng được dùng: 144
Số dòng không dùng: 2
Số CPU: 15
Số GPU: 11

Các dòng không dùng:


,RunId,ReviewStatus,TestName,Metric
19,RUN-GAMING-020,REVIEW,Time Spy,Overall
31,RUN-GAMING-032,REVIEW,Time Spy,Overall



Ví dụ dữ liệu đã ghép điểm:


,RunId,CpuModel,GpuModel,CpuMultiScore,GpuScore,Score
0,RUN-GAMING-001,AMD Ryzen 5 5500,NVIDIA GeForce RTX 3050 8GB,19252.0,12460.0,7185.0
1,RUN-GAMING-002,AMD Ryzen 5 5500,NVIDIA GeForce RTX 3060 12GB,19252.0,16710.0,9680.0
2,RUN-GAMING-003,AMD Ryzen 5 5500,NVIDIA GeForce RTX 4060 Ti 8 GB,19252.0,22596.0,13197.0
3,RUN-GAMING-004,AMD Ryzen 5 5500,NVIDIA GeForce RTX 4070 SUPER,19252.0,29942.0,17775.0
4,RUN-GAMING-005,AMD Ryzen 5 5500,NVIDIA GeForce RTX 4080 SUPER,19252.0,34225.0,20502.0



Khoảng điểm trong dữ liệu:


,GpuScore,CpuMultiScore,Score
min,12460.0,13953.0,6838.0
max,35623.0,70109.0,35820.0


## 3. Huấn luyện mô hình hồi quy log–log

Sử dụng hai biến đầu vào `GpuScore` và `CpuMultiScore` để dự đoán `Score` — điểm Time Spy Overall.

Mô hình học phương trình:

`ln(Score) = a + b × ln(GpuScore) + c × ln(CpuMultiScore)`

Hồi quy tuyến tính được ràng buộc hệ số `b, c ≥ 0`. Khi chuyển về thang điểm gốc:

`Score = exp(a) × GpuScore^b × CpuMultiScore^c`

Các bước lấy log đầu vào và đổi ngược kết quả được đóng gói cùng mô hình. Khi sử dụng, truyền điểm đầu vào gốc vào `predict()`; không lấy log hoặc exp thêm lần nữa.

Huấn luyện trên toàn bộ dữ liệu hợp lệ sau khi đã chọn phương án ở notebook so sánh. Bước này không chia lại train/test và không dùng sai số huấn luyện để khẳng định khả năng dự đoán dữ liệu mới.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# Thứ tự đầu vào được dùng thống nhất khi train và dự đoán.
FEATURES = ["GpuScore", "CpuMultiScore"]
TARGET = "Score"

X = data[FEATURES].copy()
y = data[TARGET].copy()

# Lấy log đầu vào, sau đó học hồi quy có hệ số không âm.
input_pipeline = Pipeline([
    ("log_input", FunctionTransformer(np.log, validate=True)),
    ("linear", LinearRegression(positive=True))
])

# Tự lấy log điểm đích khi train và exp kết quả khi predict.
final_model = TransformedTargetRegressor(
    regressor=input_pipeline,
    func=np.log,
    inverse_func=np.exp
)

final_model.fit(X, y)

# Lấy hệ số từ mô hình đã fit.
linear_model = final_model.regressor_.named_steps["linear"]

a = float(linear_model.intercept_)
b = float(linear_model.coef_[0])  # Hệ số log GPU
c = float(linear_model.coef_[1])  # Hệ số log CPU
A = float(np.exp(a))

if not np.isfinite([a, A, b, c]).all() or A <= 0:
    raise ValueError("Hệ số mô hình không hợp lệ.")

if b < 0 or c < 0:
    raise ValueError("Hệ số không đáp ứng ràng buộc không âm.")

print("Đã huấn luyện trên", len(data), "dòng.")
print("Thứ tự đầu vào:", FEATURES)

display(pd.DataFrame({
    "Hệ số": ["a — hệ số chặn trên thang log", "A = exp(a)",
              "b — số mũ GPU", "c — số mũ CPU"],
    "Giá trị": [a, A, b, c]
}))

print("\nCông thức trên thang điểm gốc:")
print(
    f"Score = {A:.12g}"
    f" × GpuScore^{b:.12g}"
    f" × CpuMultiScore^{c:.12g}"
)

# Kiểm tra công thức xuất cho backend khớp predict().
pred_model = final_model.predict(X)
pred_formula = (
    A
    * X["GpuScore"].to_numpy() ** b
    * X["CpuMultiScore"].to_numpy() ** c
)

if not np.isfinite(pred_model).all() or (pred_model <= 0).any():
    raise ValueError("Mô hình tạo ra điểm dự đoán không hợp lệ.")

np.testing.assert_allclose(
    pred_model,
    pred_formula,
    rtol=1e-10,
    atol=1e-8
)

print("\nĐạt: công thức tính trực tiếp khớp với model.predict().")
print("Đạt: tăng từng điểm đầu vào không làm giảm điểm dự đoán.")

Đã huấn luyện trên 144 dòng.
Thứ tự đầu vào: ['GpuScore', 'CpuMultiScore']


,Hệ số,Giá trị
0,a — hệ số chặn trên thang log,-5.033997
1,A = exp(a),0.006513
2,b — số mũ GPU,1.239672
3,c — số mũ CPU,0.216177



Công thức trên thang điểm gốc:
Score = 0.00651272831428 × GpuScore^1.23967158746 × CpuMultiScore^0.216177069224

Đạt: công thức tính trực tiếp khớp với model.predict().
Đạt: tăng từng điểm đầu vào không làm giảm điểm dự đoán.


## 4. Kiểm tra mô hình sau huấn luyện

Thực hiện hai kiểm tra:

1. Tính dự đoán và sai số trên dữ liệu đã dùng để huấn luyện, hiển thị những dòng lệch nhiều nhất để kiểm tra.
2. Tạo các tổ hợp từ danh mục CPU/GPU trong dữ liệu, kiểm tra điểm dự đoán không giảm khi tăng CPU Mark hoặc G3D Mark, giữ điểm còn lại cố định.

Các tổ hợp tạo thêm chỉ phục vụ kiểm tra hành vi của công thức, không phải benchmark thực đo và không được bổ sung vào dữ liệu huấn luyện.

MAE, RMSE và R² ở bước này là chỉ số trên tập huấn luyện. Chúng không thay thế kết quả CV/test ở notebook so sánh và không chứng minh độ chính xác trên dữ liệu mới.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Dự đoán trên dữ liệu đã dùng để train.
training_predictions = data.copy()
training_predictions["Predicted"] = final_model.predict(data[FEATURES])

training_predictions["Error"] = (
    training_predictions["Predicted"] - training_predictions["Score"]
)
training_predictions["AbsoluteError"] = (
    training_predictions["Error"].abs()
)
training_predictions["AbsolutePercentageError"] = (
    100
    * training_predictions["AbsoluteError"]
    / training_predictions["Score"]
)

training_metrics = {
    "MAE": float(mean_absolute_error(
        data["Score"], training_predictions["Predicted"]
    )),
    "RMSE": float(np.sqrt(mean_squared_error(
        data["Score"], training_predictions["Predicted"]
    ))),
    "R2": float(r2_score(
        data["Score"], training_predictions["Predicted"]
    ))
}

print("Chỉ số trên tập HUẤN LUYỆN, không phải test:")
display(pd.DataFrame([training_metrics]))

print("\n10 dòng có sai số tuyệt đối lớn nhất:")
display(
    training_predictions.nlargest(10, "AbsoluteError")[
        ["RunId", "CpuModel", "GpuModel", "Score", "Predicted",
         "AbsoluteError", "AbsolutePercentageError"]
    ].round(2)
)


# 2. Tạo các tổ hợp CPU/GPU để kiểm tra công thức.
cpu_catalog = (
    data[["CpuKey", "CpuModel", "CpuMultiScore"]]
    .drop_duplicates("CpuKey")
    .sort_values("CpuMultiScore")
)

gpu_catalog = (
    data[["GpuKey", "GpuModel", "GpuScore"]]
    .drop_duplicates("GpuKey")
    .sort_values("GpuScore")
)

prediction_grid = cpu_catalog.merge(gpu_catalog, how="cross")
prediction_grid["Predicted"] = final_model.predict(
    prediction_grid[FEATURES]
)

if (
    not np.isfinite(prediction_grid["Predicted"]).all()
    or prediction_grid["Predicted"].le(0).any()
):
    raise ValueError("Có dự đoán không hữu hạn hoặc không lớn hơn 0.")

# Giữ GPU cố định, tăng CPU Mark.
for gpu_key, group in prediction_grid.groupby("GpuKey"):
    ordered = group.sort_values("CpuMultiScore")
    if (np.diff(ordered["Predicted"]) < -1e-8).any():
        raise ValueError(f"Điểm giảm khi tăng CPU Mark: {gpu_key}")

# Giữ CPU cố định, tăng GPU Score.
for cpu_key, group in prediction_grid.groupby("CpuKey"):
    ordered = group.sort_values("GpuScore")
    if (np.diff(ordered["Predicted"]) < -1e-8).any():
        raise ValueError(f"Điểm giảm khi tăng GPU Score: {cpu_key}")

print(f"\nĐạt: kiểm tra {len(prediction_grid)} tổ hợp CPU–GPU.")
print("Không phát hiện điểm giảm khi tăng từng điểm đầu vào.")


# 3. Minh họa thay CPU với GPU có G3D Mark thấp nhất.
weakest_gpu_key = gpu_catalog.iloc[0]["GpuKey"]
example = prediction_grid.loc[
    prediction_grid["GpuKey"].eq(weakest_gpu_key)
].sort_values("CpuMultiScore")

print("\nMinh họa giữ GPU cố định, thay CPU:")
display(
    example[
        ["GpuModel", "CpuModel", "CpuMultiScore", "Predicted"]
    ].round(2)
)

Chỉ số trên tập HUẤN LUYỆN, không phải test:


,MAE,RMSE,R2
0,1525.958555,1956.811409,0.926115



10 dòng có sai số tuyệt đối lớn nhất:


,RunId,CpuModel,GpuModel,Score,Predicted,AbsoluteError,AbsolutePercentageError
129,RUN-GAMING-132,Intel Core i7 processor 14700K,NVIDIA GeForce RTX 5080,35528.0,29910.80,5617.20,15.81
109,RUN-GAMING-112,Intel Core i7 processor 14700F,NVIDIA GeForce RTX 5080,34053.0,28464.66,5588.34,16.41
137,RUN-GAMING-140,Intel Core i9 processor 14900K,NVIDIA GeForce RTX 5080,35820.0,30659.60,5160.40,14.41
110,RUN-GAMING-113,Intel Core i7 processor 14700F,NVIDIA GeForce RTX 5080,33143.0,28464.66,4678.34,14.12
108,RUN-GAMING-111,Intel Core i7 processor 14700F,NVIDIA GeForce RTX 5070 Ti,29786.0,25255.24,4530.76,15.21
98,RUN-GAMING-101,AMD Ryzen 7 7800X3D,NVIDIA GeForce RTX 5080,31820.0,27338.68,4481.32,14.08
51,RUN-GAMING-054,Intel Core i5-11600 Processor,NVIDIA GeForce RTX 5080,19519.0,23770.87,4251.87,21.78
97,RUN-GAMING-100,AMD Ryzen 7 7800X3D,NVIDIA GeForce RTX 5070 Ti,28302.0,24256.21,4045.79,14.30
136,RUN-GAMING-139,Intel Core i9 processor 14900K,NVIDIA GeForce RTX 5070 Ti,31054.0,27202.70,3851.30,12.40
135,RUN-GAMING-138,Intel Core i9 processor 14900K,NVIDIA GeForce RTX 5070,27085.0,23405.82,3679.18,13.58



Đạt: kiểm tra 165 tổ hợp CPU–GPU.
Không phát hiện điểm giảm khi tăng từng điểm đầu vào.

Minh họa giữ GPU cố định, thay CPU:


,GpuModel,CpuModel,CpuMultiScore,Predicted
0,NVIDIA GeForce RTX 3050 8GB,Intel Core i3-12100F Processor,13953.0,6121.28
11,NVIDIA GeForce RTX 3050 8GB,Intel Core i5-11600 Processor,17950.0,6463.85
22,NVIDIA GeForce RTX 3050 8GB,AMD Ryzen 5 5500,19252.0,6562.45
33,NVIDIA GeForce RTX 3050 8GB,AMD Ryzen 5 5600G,19636.0,6590.52
44,NVIDIA GeForce RTX 3050 8GB,Intel Core i5 processor 14400,25078.0,6948.43
55,NVIDIA GeForce RTX 3050 8GB,AMD Ryzen 5 9500F,28312.0,7133.03
66,NVIDIA GeForce RTX 3050 8GB,AMD Ryzen 7 7800X3D,34277.0,7434.02
77,NVIDIA GeForce RTX 3050 8GB,Intel Core i7 processor 14700F,41313.0,7740.20
88,NVIDIA GeForce RTX 3050 8GB,Intel Core Ultra 7 Processor 265F,49402.0,8045.26
99,NVIDIA GeForce RTX 3050 8GB,Intel Core i7 processor 14700K,51957.0,8133.44


## 5. Xuất mô hình và thông tin tích hợp backend

Xuất mô hình đã huấn luyện cùng công thức, hệ số đầy đủ, ý nghĩa đầu vào, khoảng dữ liệu huấn luyện và thông tin phiên bản.

- `model.joblib`: mô hình Python đã bao gồm phép biến đổi log/exp.
- `model_info.json`: hệ số để backend tính trực tiếp theo công thức `Score = A × GpuScore^b × CpuMultiScore^c`.
- `backend_test_vectors.csv`: các đầu vào và kết quả dự kiến để đối chiếu việc tính điểm ở backend.
- Các file dữ liệu và chẩn đoán giúp truy vết quá trình huấn luyện.

Trước khi xuất, đọc lại model và hệ số đã lưu, kiểm tra kết quả khớp với dự đoán trong bộ nhớ.

Backend phải nhận đúng điểm PassMark G3D Mark và CPU Mark, kiểm tra đầu vào hữu hạn và lớn hơn 0. Đầu vào ngoài khoảng huấn luyện cần được nhận diện là ngoại suy. Nằm trong khoảng của từng biến cũng không bảo đảm mọi tổ hợp phần cứng đã có trong dữ liệu.

Các chỉ số kèm theo là sai số trên tập huấn luyện. Kết quả đánh giá CV/test được giữ riêng trong notebook so sánh.

In [ ]:
import json
import zipfile
import joblib
import sklearn

from pathlib import Path
from datetime import datetime, timezone

# Tạo thư mục riêng cho lần xuất này.
created_at = datetime.now(timezone.utc)
version = created_at.strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path(f"gaming_final_{version}")
output_dir.mkdir(exist_ok=False)

# Lấy lại hệ số trực tiếp từ model, không dùng số làm tròn.
fitted_linear = final_model.regressor_.named_steps["linear"]

coefficients = {
    "A": float(np.exp(fitted_linear.intercept_)),
    "b": float(fitted_linear.coef_[0]),
    "c": float(fitted_linear.coef_[1])
}

model_info = {
    "model_version": f"gaming-log-v1-{version}",
    "created_at_utc": created_at.isoformat(),
    "model_type": "Nonnegative log-log linear regression",
    "fit_scope": "All accepted eligible rows",
    "target": "3DMark Time Spy Overall",
    "feature_order": FEATURES,
    "feature_definitions": {
        "GpuScore": "PassMark G3D Mark; exact GPU and VRAM variant",
        "CpuMultiScore": "PassMark CPU Mark"
    },
    "formula": "Score = A * GpuScore**b * CpuMultiScore**c",
    "coefficients": coefficients,
    "input_requirements": {
        "finite": True,
        "strictly_positive": True,
        "missing_values": "Reject; do not replace with zero",
        "outside_training_ranges": "Flag as extrapolation"
    },
    "training_ranges": {
        feature: {
            "min": float(data[feature].min()),
            "max": float(data[feature].max())
        }
        for feature in FEATURES
    },
    "training_row_count": len(data),
    "excluded_row_count": len(excluded_rows),
    "cpu_count": int(data["CpuKey"].nunique()),
    "gpu_count": int(data["GpuKey"].nunique()),
    "source_excel_name": excel_name,
    "source_excel_sha256": excel_sha256,
    "versions": {
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__
    },
    "training_metrics_only": training_metrics,
    "evaluation": {
        "independent_validation_of_final_model": False,
        "cv_test_results": "See separate comparison notebook and export"
    },
    "limitations": [
        "Training metrics are not independent validation.",
        "Benchmark run conditions vary.",
        "Monotonic predictions do not guarantee realistic bottlenecks.",
        "Predictions represent Time Spy Overall, not per-game FPS.",
        "Input ranges do not establish coverage of every CPU-GPU pair."
    ]
}

# 1. Lưu model và thông tin đầy đủ.
joblib.dump(final_model, output_dir / "model.joblib")

info_path = output_dir / "model_info.json"
info_path.write_text(
    json.dumps(
        model_info,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

# 2. Lưu dữ liệu và kết quả chẩn đoán, không làm tròn.
data.to_csv(
    output_dir / "data_used.csv", index=False, encoding="utf-8-sig"
)
excluded_rows.to_csv(
    output_dir / "excluded_rows.csv", index=False, encoding="utf-8-sig"
)
reference.to_csv(
    output_dir / "reference.csv", index=False, encoding="utf-8-sig"
)
training_predictions.to_csv(
    output_dir / "training_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Các tổ hợp này dùng kiểm tra phép tính, không phải dữ liệu thực đo.
backend_test_vectors = prediction_grid[
    ["CpuModel", "GpuModel"] + FEATURES + ["Predicted"]
].rename(columns={"Predicted": "ExpectedScore"})

backend_test_vectors.to_csv(
    output_dir / "backend_test_vectors.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Đọc lại model, JSON và bảng kiểm tra đã lưu.
loaded_model = joblib.load(output_dir / "model.joblib")
loaded_info = json.loads(info_path.read_text(encoding="utf-8"))
loaded_vectors = pd.read_csv(output_dir / "backend_test_vectors.csv")

expected = loaded_vectors["ExpectedScore"].to_numpy()
reloaded_predictions = loaded_model.predict(
    loaded_vectors[loaded_info["feature_order"]]
)

k = loaded_info["coefficients"]
formula_predictions = (
    k["A"]
    * loaded_vectors["GpuScore"].to_numpy() ** k["b"]
    * loaded_vectors["CpuMultiScore"].to_numpy() ** k["c"]
)

np.testing.assert_allclose(
    reloaded_predictions, expected, rtol=1e-10, atol=1e-8
)
np.testing.assert_allclose(
    formula_predictions, expected, rtol=1e-10, atol=1e-8
)

print("Đạt: model đọc lại và hệ số JSON khớp toàn bộ mẫu kiểm tra.")

# 4. Đóng gói và tải xuống.
export_names = [
    "model.joblib",
    "model_info.json",
    "data_used.csv",
    "excluded_rows.csv",
    "reference.csv",
    "training_predictions.csv",
    "backend_test_vectors.csv"
]

zip_path = output_dir.with_suffix(".zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in export_names:
        archive.write(output_dir / name, arcname=name)

print("Phiên bản:", model_info["model_version"])
print("Số dòng đã train:", len(data))
print("Gói xuất:", zip_path.name)

files.download(str(zip_path))

Đạt: model đọc lại và hệ số JSON khớp toàn bộ mẫu kiểm tra.
Phiên bản: gaming-log-v1-20260920T194416994761Z
Số dòng đã train: 144
Gói xuất: gaming_final_20260920T194416994761Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>